In [1]:
# === ByT5 XL: Setup + Load (bf16, plain HF transformers, same as mT5) ===
import os, time
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, T5EncoderModel

PROJECT_DIR = os.path.expanduser("~/Ayesha/Dyslexia/Dyslexia-ByT5")
DATA_PATH = os.path.join(PROJECT_DIR, "dyslex_probe_stimuli_clean.csv")
REPS_DIR  = os.path.join(PROJECT_DIR, "representations")
os.makedirs(REPS_DIR, exist_ok=True)

MODEL_NAME = "google/byt5-xl"   # size-matched to mt5-xl (3.7B). VRAM tight? -> "google/byt5-large"
DEVICE = "cuda"
PREFIX = "byt5"

assert os.path.exists(DATA_PATH), f"Missing CSV: {DATA_PATH}"
df = pd.read_csv(DATA_PATH)
print(f"Stimuli: {len(df)} rows, {df['language'].value_counts().to_dict()}")

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)   # ByT5Tokenizer (byte-level, no fast)

print(f"Loading {MODEL_NAME} encoder (bf16)...")
t0 = time.time()
model = T5EncoderModel.from_pretrained(
    MODEL_NAME, torch_dtype=torch.bfloat16, device_map="auto", low_cpu_mem_usage=True,
)
model.eval()
print(f"Loaded in {time.time()-t0:.1f}s")

n_layers = len(model.encoder.block)
hidden   = model.config.d_model
n_params = sum(p.numel() for p in model.parameters())
print(f"Encoder layers: {n_layers}  |  hidden: {hidden}  |  encoder params: {n_params/1e9:.2f}B")
print(f"VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB")

for s in ["The cat sat on the mat.", "বিড়াল মাদুরে বসেছিল।"]:
    ids = tokenizer(s, return_tensors="pt").input_ids[0]
    print(f"'{s}' -> {len(ids)} byte-tok, last_id={ids[-1].item()} (EOS=1)")

Stimuli: 440 rows, {'english': 220, 'bangla': 220}
Loading tokenizer...


config.json:   0%|          | 0.00/720 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.59k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.50k [00:00<?, ?B/s]

Loading google/byt5-xl encoder (bf16)...


pytorch_model.bin:   0%|          | 0.00/15.0G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/328 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/15.0G [00:00<?, ?B/s]

[transformers] T5EncoderModel LOAD REPORT from: google/byt5-xl
Key            | Status     |  | 
---------------+------------+--+-
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded in 192.5s
Encoder layers: 36  |  hidden: 2560  |  encoder params: 2.61B
VRAM: 5.23 GB
'The cat sat on the mat.' -> 24 byte-tok, last_id=1 (EOS=1)
'বিড়াল মাদুরে বসেছিল।' -> 60 byte-tok, last_id=1 (EOS=1)


In [2]:
# === ByT5: extraction + sanity ===
def extract_representation(text, model, tokenizer, device="cuda"):
    """Mean-pool encoder hidden states per layer, excluding EOS. Identical to mT5."""
    inputs = tokenizer(text, return_tensors="pt", truncation=True,
                       max_length=512, add_special_tokens=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)
    hidden_states = outputs.hidden_states            # len = n_layers+1
    attention_mask = inputs["attention_mask"][0]
    content_mask = attention_mask.clone()
    seq_len = int(attention_mask.sum().item())
    content_mask[seq_len - 1] = 0                    # zero out EOS (T5/ByT5 has no BOS)
    pooled_layers = []
    for layer_hidden in hidden_states:
        layer = layer_hidden[0].float()
        masked = layer * content_mask.unsqueeze(-1).float()
        denom = content_mask.sum().clamp(min=1).float()
        pooled_layers.append((masked.sum(dim=0) / denom).cpu().numpy())
    return np.stack(pooled_layers)                   # (n_layers+1, hidden)

for _, row in df.sample(5, random_state=42).iterrows():
    reps = extract_representation(str(row['word']), model, tokenizer, DEVICE)
    print(f"[{row['language']}] '{str(row['word'])[:30]}' -> shape {reps.shape}, "
          f"mean={reps.mean():.4f}, std={reps.std():.4f}")
print(f"VRAM after sanity: {torch.cuda.memory_allocated()/1e9:.2f} GB")

[bangla] 'ভালবাসা' -> shape (37, 2560), mean=-2.7321, std=191.6105
[bangla] 'জাম' -> shape (37, 2560), mean=-2.8327, std=175.4102
[english] 'why' -> shape (37, 2560), mean=-2.7951, std=162.3090
[bangla] 'বাড়ি' -> shape (37, 2560), mean=-2.8244, std=180.1103
[bangla] 'মধ্য' -> shape (37, 2560), mean=-2.7881, std=174.1094
VRAM after sanity: 5.26 GB


In [3]:
# === ByT5: full extraction across 440 stimuli + save ===
from tqdm.auto import tqdm
all_reps, langs, ids = [], [], []
t0 = time.time()
for _, row in tqdm(df.iterrows(), total=len(df), desc="Extracting"):
    all_reps.append(extract_representation(str(row['word']), model, tokenizer, DEVICE))
    langs.append(row['language']); ids.append(row['id'])

all_reps = np.stack(all_reps); langs = np.array(langs); ids = np.array(ids)
print(f"Done in {time.time()-t0:.1f}s | all_reps shape: {all_reps.shape}")

en_mask = (langs == "english"); bn_mask = (langs == "bangla")
reps_en, reps_bn = all_reps[en_mask], all_reps[bn_mask]
np.save(os.path.join(REPS_DIR, f"{PREFIX}_en_reps.npy"), reps_en)
np.save(os.path.join(REPS_DIR, f"{PREFIX}_bn_reps.npy"), reps_bn)
np.save(os.path.join(REPS_DIR, f"{PREFIX}_ids_en.npy"), ids[en_mask])
np.save(os.path.join(REPS_DIR, f"{PREFIX}_ids_bn.npy"), ids[bn_mask])
print(f"Saved: {PREFIX}_en_reps {reps_en.shape}, {PREFIX}_bn_reps {reps_bn.shape}")
print(f"NaN check: en={np.isnan(reps_en).any()}, bn={np.isnan(reps_bn).any()}")

Extracting:   0%|          | 0/440 [00:00<?, ?it/s]

Done in 10.8s | all_reps shape: (440, 37, 2560)
Saved: byt5_en_reps (220, 37, 2560), byt5_bn_reps (220, 37, 2560)
NaN check: en=False, bn=False


In [4]:
# === ByT5 RQ1: Per-layer English probing (5-fold CV) ===
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.metrics import f1_score, r2_score

df_en = df[df['language'] == 'english'].reset_index(drop=True)
assert len(df_en) == reps_en.shape[0], "Row count mismatch EN"
N_LAYERS = reps_en.shape[1]   # 37 = embeddings + 36

def build_task_masks_labels(df_lang):
    tasks = {}
    m = df_lang['voiced_unvoiced'].isin(['voiced','unvoiced']).values
    y = (df_lang.loc[m, 'voiced_unvoiced'] == 'voiced').astype(int).values
    tasks['voicing'] = (m, y, 'binary')
    m = df_lang['is_real_word'].isin([True, False, 'True', 'False', 'true','false', 1, 0]).values
    y = df_lang.loc[m, 'is_real_word'].astype(str).str.lower().isin(['true','1']).astype(int).values
    tasks['realword'] = (m, y, 'binary')
    m = df_lang['orthographic_transparency'].isin(['transparent','opaque']).values
    y = (df_lang.loc[m, 'orthographic_transparency'] == 'transparent').astype(int).values
    tasks['ortho'] = (m, y, 'binary')
    m = df_lang['syllable_count'].notna().values
    y = df_lang.loc[m, 'syllable_count'].astype(float).values
    tasks['syllable'] = (m, y, 'regression')
    return tasks

tasks_en = build_task_masks_labels(df_en)
for tname, (m, y, kind) in tasks_en.items():
    print(f"{tname:10s}: n={m.sum():3d}  kind={kind}  "
          + (f"pos={y.sum()}/{len(y)}" if kind=='binary' else f"range={y.min():.0f}-{y.max():.0f}"))

def cv_score(X, y, kind, n_splits=5):
    if kind == 'binary':
        if len(np.unique(y)) < 2 or min(np.bincount(y)) < n_splits: return np.nan
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
        s = []
        for tr, te in skf.split(X, y):
            clf = make_pipeline(StandardScaler(),
                                LogisticRegression(max_iter=2000, C=1.0, solver='liblinear'))
            clf.fit(X[tr], y[tr]); s.append(f1_score(y[te], clf.predict(X[te]), average='macro'))
        return float(np.mean(s))
    else:
        kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
        s = []
        for tr, te in kf.split(X):
            reg = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
            reg.fit(X[tr], y[tr]); s.append(r2_score(y[te], reg.predict(X[te])))
        return float(np.mean(s))

rows = []
for tname, (m, y, kind) in tasks_en.items():
    for L in range(N_LAYERS):
        rows.append({'task': tname, 'layer': L, 'kind': kind,
                     'score': cv_score(reps_en[m, L, :], y, kind)})
res_en = pd.DataFrame(rows)
res_en.to_csv(os.path.join(PROJECT_DIR, "results_english_probing_byt5.csv"), index=False)

print("\nRQ1 peaks (ByT5 EN):")
for t in ['voicing','realword','ortho','syllable']:
    sub = res_en[res_en.task==t].dropna()
    if len(sub):
        peak = sub.loc[sub.score.idxmax()]
        print(f"  {t:10s}: L{int(peak.layer):>2d}/{N_LAYERS-1}  rel={peak.layer/(N_LAYERS-1):.2f}  score={peak.score:.3f}")

voicing   : n= 30  kind=binary  pos=15/30
realword  : n=220  kind=binary  pos=195/220
ortho     : n=220  kind=binary  pos=182/220
syllable  : n=220  kind=regression  range=1-8

RQ1 peaks (ByT5 EN):
  voicing   : L 1/36  rel=0.03  score=0.966
  realword  : L23/36  rel=0.64  score=0.722
  ortho     : L 9/36  rel=0.25  score=0.723
  syllable  : L28/36  rel=0.78  score=0.903


In [5]:
# === ByT5 RQ2: Cross-lingual transfer EN -> BN (frozen probe) ===
df_bn = df[df['language'] == 'bangla'].reset_index(drop=True)
assert len(df_bn) == reps_bn.shape[0]

# Bangla voicing label fix (first-char inference) -- same as mT5 run
voiced_starters   = {'গ','ব','দ','ড','জ','ঘ','ভ','ধ','ঢ','ঝ'}
unvoiced_starters = {'ক','প','ত','ট','চ','খ','ফ','থ','ঠ','ছ'}
def infer_bn_voicing(row):
    if row.get('error_type_targeted') not in ('voiced_unvoiced_confusion','aspiration_confusion'):
        return row.get('voiced_unvoiced')
    fc = str(row['word'])[0] if str(row['word']) else ''
    if fc in voiced_starters:   return 'voiced'
    if fc in unvoiced_starters: return 'unvoiced'
    return 'na'
df_bn = df_bn.copy()
df_bn['voiced_unvoiced'] = df_bn.apply(infer_bn_voicing, axis=1)

tasks_bn = build_task_masks_labels(df_bn)

def transfer_score(X_tr, y_tr, X_te, y_te, kind):
    if kind == 'binary':
        if len(np.unique(y_tr)) < 2 or len(np.unique(y_te)) < 2: return np.nan
        clf = make_pipeline(StandardScaler(),
                            LogisticRegression(max_iter=2000, C=1.0, solver='liblinear'))
        clf.fit(X_tr, y_tr); return float(f1_score(y_te, clf.predict(X_te), average='macro'))
    else:
        reg = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
        reg.fit(X_tr, y_tr); return float(r2_score(y_te, reg.predict(X_te)))

rows = []
for tname in ['voicing','realword','ortho','syllable']:
    m_en, y_en, kind = tasks_en[tname]
    m_bn, y_bn, _    = tasks_bn[tname]
    if m_bn.sum() == 0: continue
    for L in range(N_LAYERS):
        rows.append({'task': tname, 'layer': L, 'kind': kind,
                     'bn_score': transfer_score(reps_en[m_en, L, :], y_en,
                                                reps_bn[m_bn, L, :], y_bn, kind)})
res_tr = pd.DataFrame(rows)
res_tr.to_csv(os.path.join(PROJECT_DIR, "results_transfer_byt5.csv"), index=False)

print("\nRQ2 EN->BN (ByT5):")
print(f"{'task':10s} {'best_L':>6s} {'rel':>5s} {'BN':>6s} {'EN_peak':>8s} {'ratio':>6s}")
for t in ['voicing','realword','ortho','syllable']:
    sub = res_tr[res_tr.task==t].dropna()
    if len(sub)==0: continue
    best = sub.loc[sub.bn_score.idxmax()]
    en_peak = res_en[res_en.task==t].score.max()
    ratio = best.bn_score/en_peak if en_peak>0 else np.nan
    print(f"{t:10s} {int(best.layer):>6d} {best.layer/(N_LAYERS-1):>5.2f} {best.bn_score:>6.3f} {en_peak:>8.3f} {ratio:>6.3f}")


RQ2 EN->BN (ByT5):
task       best_L   rel     BN  EN_peak  ratio
voicing        10  0.28  0.650    0.966  0.673
realword       30  0.83  0.750    0.722  1.040
ortho           2  0.06  0.711    0.723  0.983
syllable       34  0.94 -0.019    0.903 -0.021


In [6]:
# === ByT5: Length-matched realword sanity + transfer (mirrors mT5 CELL 19/20) ===
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import f1_score
from collections import defaultdict

def cv_score_bin(X, y, n_splits=5):
    if len(np.unique(y)) < 2 or min(np.bincount(y)) < n_splits: return np.nan
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    s = []
    for tr, te in skf.split(X, y):
        clf = make_pipeline(StandardScaler(),
                            LogisticRegression(max_iter=2000, C=1.0, solver='liblinear'))
        clf.fit(X[tr], y[tr]); s.append(f1_score(y[te], clf.predict(X[te]), average='macro'))
    return float(np.mean(s))

def build_matched(df_lang, mask, seed=42):
    rng = np.random.default_rng(seed)
    positions_in_reps = np.where(mask)[0]
    chars = df_lang.iloc[positions_in_reps]['word'].astype(str).str.len().values
    is_real = df_lang.iloc[positions_in_reps]['is_real_word'].astype(str).str.lower().isin(['true','1']).values
    real_by_len, pseudo_by_len = defaultdict(list), defaultdict(list)
    for local_i, (cl, ir) in enumerate(zip(chars, is_real)):
        gp = positions_in_reps[local_i]
        (real_by_len if ir else pseudo_by_len)[cl].append(gp)
    out_pos, out_y = [], []
    for L, reals in real_by_len.items():
        pseudos = pseudo_by_len.get(L, [])
        k = min(len(reals), len(pseudos))
        if k == 0: continue
        rng.shuffle(reals); rng.shuffle(pseudos)
        for p in reals[:k]:   out_pos.append(p); out_y.append(1)
        for p in pseudos[:k]: out_pos.append(p); out_y.append(0)
    return np.array(out_pos), np.array(out_y)

def surface_on_matched(df_lang, positions):
    return np.array([[len(str(df_lang.iloc[p]['word'])),
                      len(set(str(df_lang.iloc[p]['word']))),
                      len(str(df_lang.iloc[p]['word']).encode('utf-8'))] for p in positions], dtype=float)

m_en, _, _ = tasks_en['realword']
m_bn, _, _ = tasks_bn['realword']
pos_en, y_en_match = build_matched(df_en, m_en)
pos_bn, y_bn_match = build_matched(df_bn, m_bn)
print(f"EN matched: n={len(pos_en)}  real={y_en_match.sum()} pseudo={len(y_en_match)-y_en_match.sum()}")
print(f"BN matched: n={len(pos_bn)}  real={y_bn_match.sum()} pseudo={len(y_bn_match)-y_bn_match.sum()}")
print(f"EN mean char len={np.mean([len(str(df_en.iloc[p]['word'])) for p in pos_en]):.2f} (mt5 run: 6.88)")
print(f"BN mean char len={np.mean([len(str(df_bn.iloc[p]['word'])) for p in pos_bn]):.2f} (mt5 run: 5.20)")

surf_en = cv_score_bin(surface_on_matched(df_en, pos_en), y_en_match)
surf_bn = cv_score_bin(surface_on_matched(df_bn, pos_bn), y_bn_match)
print(f"\nSurface baseline  EN={surf_en:.3f}  BN={surf_bn:.3f}   <-- should match mt5 run: 0.666 / 0.623")

# within-language L0 / peak / gain
sc_en = [cv_score_bin(reps_en[pos_en, L, :], y_en_match) for L in range(N_LAYERS)]
sc_bn = [cv_score_bin(reps_bn[pos_bn, L, :], y_bn_match) for L in range(N_LAYERS)]
l0_en, pk_en = sc_en[0], np.nanmax(sc_en)
l0_bn, pk_bn = sc_bn[0], np.nanmax(sc_bn)
gain_en = pk_en - max(surf_en, l0_en); gain_bn = pk_bn - max(surf_bn, l0_bn)
print("\n=== ByT5 within-language realword (length-matched) ===")
print(f"  EN  L0={l0_en:.3f}  peak={pk_en:.3f} (L{int(np.nanargmax(sc_en))}/{N_LAYERS-1})  gain={gain_en:+.3f}")
print(f"  BN  L0={l0_bn:.3f}  peak={pk_bn:.3f} (L{int(np.nanargmax(sc_bn))}/{N_LAYERS-1})  gain={gain_bn:+.3f}")
pd.DataFrame([{'model':'byt5','L0_EN':l0_en,'peak_EN':pk_en,'gain_EN':gain_en,
               'L0_BN':l0_bn,'peak_BN':pk_bn,'gain_BN':gain_bn,
               'surface_EN':surf_en,'surface_BN':surf_bn,
               'n_en':len(pos_en),'n_bn':len(pos_bn)}]
            ).to_csv(os.path.join(PROJECT_DIR,"results_realword_length_matched_byt5.csv"), index=False)

# EN->BN transfer, per layer
def transfer_bin(Xtr, ytr, Xte, yte):
    if len(np.unique(ytr))<2 or len(np.unique(yte))<2: return np.nan
    clf = make_pipeline(StandardScaler(),
                        LogisticRegression(max_iter=2000, C=1.0, solver='liblinear'))
    clf.fit(Xtr, ytr); return float(f1_score(yte, clf.predict(Xte), average='macro'))

rows = [{'model':'byt5','layer':L,'rel_depth':L/max(N_LAYERS-1,1),
         'bn_transfer_f1':transfer_bin(reps_en[pos_en,L,:],y_en_match,reps_bn[pos_bn,L,:],y_bn_match),
         'n_layers':N_LAYERS} for L in range(N_LAYERS)]
tr_df = pd.DataFrame(rows); tr_df.to_csv(os.path.join(PROJECT_DIR,"results_transfer_realword_matched_byt5.csv"), index=False)
pk = tr_df.dropna(subset=['bn_transfer_f1']).sort_values('bn_transfer_f1').iloc[-1]
print("\n=== ByT5 EN->BN transfer (length-matched) ===")
print(f"  peak L{int(pk.layer)}/{N_LAYERS-1}  rel={pk.rel_depth:.2f}  BN_F1={pk.bn_transfer_f1:.3f}  "
      f"gain_vs_surface={pk.bn_transfer_f1 - surf_bn:+.3f}")

EN matched: n=50  real=25 pseudo=25
BN matched: n=50  real=25 pseudo=25
EN mean char len=6.88 (mt5 run: 6.88)
BN mean char len=5.20 (mt5 run: 5.20)

Surface baseline  EN=0.666  BN=0.623   <-- should match mt5 run: 0.666 / 0.623

=== ByT5 within-language realword (length-matched) ===
  EN  L0=0.571  peak=0.796 (L22/36)  gain=+0.130
  BN  L0=0.647  peak=0.918 (L28/36)  gain=+0.271

=== ByT5 EN->BN transfer (length-matched) ===
  peak L29/36  rel=0.81  BN_F1=0.920  gain_vs_surface=+0.297


In [7]:
# === Per-layer EN->BN transfer emergence: byte vs subword ===
import matplotlib.pyplot as plt
import pandas as pd, numpy as np, os

BASE = os.path.expanduser("~/Ayesha/Dyslexia")
tr4 = pd.read_csv(f"{BASE}/dyslexia-mt5/results_transfer_realword_matched.csv")
trB = pd.read_csv(f"{BASE}/Dyslexia-ByT5/results_transfer_realword_matched_byt5.csv")
allt = pd.concat([tr4, trB], ignore_index=True)
SURF_BN = 0.623

COLORS = {'xlmr':'#2F6FA6','bloom':'#3A9A55','mt5':'#8A5CB0','llama':'#C33F3C','byt5':'#E07B00'}
LABELS = {'xlmr':'XLM-R (subword enc)','bloom':'BLOOM (subword dec)',
          'mt5':'mT5 (subword enc-dec)','llama':'Llama (subword dec)','byt5':'ByT5 (BYTE enc-dec)'}
EMPH = {'mt5','byt5'}

fig, ax = plt.subplots(figsize=(9, 5.5))
for m in ['xlmr','llama','bloom','mt5','byt5']:
    sub = allt[allt.model==m].dropna(subset=['bn_transfer_f1']).sort_values('rel_depth')
    e = m in EMPH
    ax.plot(sub.rel_depth, sub.bn_transfer_f1, marker='o', ms=5 if e else 3,
            lw=3 if e else 1.4, alpha=1.0 if e else 0.45, color=COLORS[m],
            label=LABELS[m], zorder=3 if e else 2)
ax.axhline(SURF_BN, color='gray', ls='--', lw=1.2, alpha=0.8, label=f'surface baseline ({SURF_BN:.2f})')
ax.axhline(0.5, color='black', ls='-', lw=0.6, alpha=0.25)
ax.set_xlabel('relative encoder depth', fontsize=11)
ax.set_ylabel('EN→BN transfer F1 (macro, length-matched)', fontsize=11)
ax.set_title('Cross-lingual lexical transfer by depth:\nbyte-level ByT5 survives where subword mT5 collapses',
             fontsize=11.5, fontweight='bold')
ax.set_ylim(0.45, 1.0); ax.grid(alpha=0.3); ax.legend(fontsize=8.5, loc='lower right')
for m, dy in [('byt5', 0.025), ('mt5', -0.05)]:
    sub = allt[allt.model==m].dropna(subset=['bn_transfer_f1'])
    pk = sub.loc[sub.bn_transfer_f1.idxmax()]
    ax.annotate(f"{m}\n{pk.bn_transfer_f1:.2f}", xy=(pk.rel_depth, pk.bn_transfer_f1),
                xytext=(pk.rel_depth, pk.bn_transfer_f1+dy), fontsize=9,
                fontweight='bold', color=COLORS[m], ha='center')
fig.tight_layout()
out = f"{BASE}/Dyslexia-ByT5/plots_byt5_vs_mt5_transfer_emergence"
fig.savefig(out+".png", dpi=130, bbox_inches='tight'); fig.savefig(out+".pdf", bbox_inches='tight')
plt.close(fig)
print("Saved", out+".png / .pdf\n")
print("Peak transfer by model:")
for m in ['xlmr','llama','bloom','mt5','byt5']:
    sub = allt[allt.model==m].dropna(subset=['bn_transfer_f1'])
    pk = sub.loc[sub.bn_transfer_f1.idxmax()]
    print(f"  {m:5s}  peak rel={pk.rel_depth:.2f}  BN_F1={pk.bn_transfer_f1:.3f}  gain={pk.bn_transfer_f1-SURF_BN:+.3f}")

Saved /home/accesx-denied/Ayesha/Dyslexia/Dyslexia-ByT5/plots_byt5_vs_mt5_transfer_emergence.png / .pdf

Peak transfer by model:
  xlmr   peak rel=0.67  BN_F1=0.859  gain=+0.236
  llama  peak rel=0.97  BN_F1=0.635  gain=+0.012
  bloom  peak rel=0.83  BN_F1=0.920  gain=+0.297
  mt5    peak rel=0.83  BN_F1=0.599  gain=-0.024
  byt5   peak rel=0.81  BN_F1=0.920  gain=+0.297


In [8]:
# === ByT5 RQ3: ablation on length-matched realword (mirrors mT5 CELL 23) ===
import time
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import f1_score

try:
    _ = model.encoder.block[0]; HIDDEN = model.config.d_model
    print(f"ByT5 loaded. hidden={HIDDEN}  VRAM={torch.cuda.memory_allocated()/1e9:.2f} GB")
except Exception as e:
    raise RuntimeError(f"ByT5 not loaded -- re-run Cell 1. ({e})")

MEASURE_L = 29                 # ByT5 length-matched transfer peak
ABLATE_BLOCK = MEASURE_L - 3   # 26
print(f"measure_L={MEASURE_L}  ablate_block={ABLATE_BLOCK}")

pipe = make_pipeline(StandardScaler(),
                     LogisticRegression(max_iter=2000, C=1.0, solver='liblinear'))
pipe.fit(reps_en[pos_en, MEASURE_L, :], y_en_match)
ranked = np.argsort(np.abs(pipe.named_steps['logisticregression'].coef_[0]))[::-1].copy()
print(f"Top-5 ranked units: {ranked[:5].tolist()}")
base_en = f1_score(y_en_match, pipe.predict(reps_en[pos_en, MEASURE_L, :]), average='macro')
base_bn = f1_score(y_bn_match, pipe.predict(reps_bn[pos_bn, MEASURE_L, :]), average='macro')
print(f"Baseline: EN={base_en:.3f}  BN={base_bn:.3f}  (EN~1.0 expected)")

texts_en = df_en.iloc[pos_en]['word'].astype(str).tolist()
texts_bn = df_bn.iloc[pos_bn]['word'].astype(str).tolist()

def extract_with_ablation_byt5(text, ablate_block, ablate_units):
    units_t = torch.tensor(ablate_units, dtype=torch.long, device=DEVICE)
    def hook_fn(module, inp, output):
        if isinstance(output, tuple):
            h = output[0].clone(); h[:, :, units_t] = 0.0; return (h,) + output[1:]
        h = output.clone(); h[:, :, units_t] = 0.0; return h
    handle = model.encoder.block[ablate_block].register_forward_hook(hook_fn)
    try:
        inp = tokenizer(text, return_tensors="pt", truncation=True, max_length=512, add_special_tokens=True)
        inp = {k: v.to(DEVICE) for k, v in inp.items()}
        with torch.no_grad():
            out = model(**inp, output_hidden_states=True)
        am = inp["attention_mask"][0]; cm = am.clone(); cm[int(am.sum().item())-1] = 0
        pooled = []
        for lh in out.hidden_states:
            layer = lh[0].float(); masked = layer * cm.unsqueeze(-1).float()
            pooled.append((masked.sum(0) / cm.sum().clamp(min=1).float()).cpu().numpy())
        return np.stack(pooled)
    finally:
        handle.remove()

rng = np.random.default_rng(42)
CONDITIONS = {"top5": ranked[:5].tolist(), "top10": ranked[:10].tolist(),
              "top20": ranked[:20].tolist(), "rand10": rng.choice(HIDDEN, size=10, replace=False).tolist()}

print("\n=== ByT5 ablation (frozen probe on matched EN baseline) ===")
rows = [{'model':'byt5','condition':'baseline','n_units':0,'en_score':base_en,'bn_score':base_bn}]
t0 = time.time()
for cond, units in CONDITIONS.items():
    en_abl = np.stack([extract_with_ablation_byt5(t, ABLATE_BLOCK, units) for t in texts_en])
    bn_abl = np.stack([extract_with_ablation_byt5(t, ABLATE_BLOCK, units) for t in texts_bn])
    s_en = f1_score(y_en_match, pipe.predict(en_abl[:, MEASURE_L, :]), average='macro')
    s_bn = f1_score(y_bn_match, pipe.predict(bn_abl[:, MEASURE_L, :]), average='macro')
    print(f"  {cond:7s} ({len(units):2d}u)  EN={s_en:.3f} (Δ{base_en-s_en:+.3f})  BN={s_bn:.3f} (Δ{base_bn-s_bn:+.3f})")
    rows.append({'model':'byt5','condition':cond,'n_units':len(units),'en_score':s_en,'bn_score':s_bn})
print(f"Extraction time: {time.time()-t0:.1f}s")

abl = pd.DataFrame(rows); abl.to_csv(os.path.join(PROJECT_DIR, "results_ablation_byt5_realword_matched.csv"), index=False)
sub = abl.set_index('condition')
spec_en = (base_en - sub.loc['top20','en_score']) - (base_en - sub.loc['rand10','en_score'])
spec_bn = (base_bn - sub.loc['top20','bn_score']) - (base_bn - sub.loc['rand10','bn_score'])
print(f"\nSpecificity (top20 drop - rand10 drop):  EN={spec_en:+.3f}  BN={spec_bn:+.3f}")
print("Reference -> mt5 BN spec=+0.11 (localized) | bloom -0.02 (distributed) | xlmr +0.004")

ByT5 loaded. hidden=2560  VRAM=5.26 GB
measure_L=29  ablate_block=26
Top-5 ranked units: [1675, 108, 1641, 1194, 2022]
Baseline: EN=1.000  BN=0.920  (EN~1.0 expected)

=== ByT5 ablation (frozen probe on matched EN baseline) ===
  top5    ( 5u)  EN=1.000 (Δ+0.000)  BN=0.920 (Δ+0.000)
  top10   (10u)  EN=1.000 (Δ+0.000)  BN=0.920 (Δ+0.000)
  top20   (20u)  EN=1.000 (Δ+0.000)  BN=0.920 (Δ+0.000)
  rand10  (10u)  EN=1.000 (Δ+0.000)  BN=0.920 (Δ+0.000)
Extraction time: 9.4s

Specificity (top20 drop - rand10 drop):  EN=+0.000  BN=+0.000
Reference -> mt5 BN spec=+0.11 (localized) | bloom -0.02 (distributed) | xlmr +0.004


In [9]:
# === ByT5 ablation diagnostic: zero top-K units DIRECTLY at measure layer L29 ===
# Resolves "forward-hook washout" vs "genuinely distributed".
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import f1_score

L = 29
Xen = reps_en[pos_en, L, :].copy()
Xbn = reps_bn[pos_bn, L, :].copy()
pipe = make_pipeline(StandardScaler(),
                     LogisticRegression(max_iter=2000, C=1.0, solver='liblinear'))
pipe.fit(Xen, y_en_match)
ranked = np.argsort(np.abs(pipe.named_steps['logisticregression'].coef_[0]))[::-1].copy()
base_en = f1_score(y_en_match, pipe.predict(Xen), average='macro')
base_bn = f1_score(y_bn_match, pipe.predict(Xbn), average='macro')
print(f"baseline EN={base_en:.3f}  BN={base_bn:.3f}   (hidden dim = {Xen.shape[1]})\n")
print(f"{'K':>4s} | {'topK_EN':>7s} {'topK_BN':>7s} | {'randK_EN':>8s} {'randK_BN':>8s}")
rng = np.random.default_rng(42)
for K in [5, 10, 20, 50, 100, 200, 400]:
    top = ranked[:K]; rnd = rng.choice(Xen.shape[1], size=K, replace=False)
    def z(X, idx):
        Xc = X.copy(); Xc[:, idx] = 0.0; return Xc
    te = f1_score(y_en_match, pipe.predict(z(Xen, top)), average='macro')
    tb = f1_score(y_bn_match, pipe.predict(z(Xbn, top)), average='macro')
    re_ = f1_score(y_en_match, pipe.predict(z(Xen, rnd)), average='macro')
    rb = f1_score(y_bn_match, pipe.predict(z(Xbn, rnd)), average='macro')
    print(f"{K:>4d} | {te:>7.3f} {tb:>7.3f} | {re_:>8.3f} {rb:>8.3f}")

baseline EN=1.000  BN=0.920   (hidden dim = 2560)

   K | topK_EN topK_BN | randK_EN randK_BN
   5 |   1.000   0.900 |    1.000    0.920
  10 |   1.000   0.920 |    1.000    0.920
  20 |   1.000   0.900 |    1.000    0.920
  50 |   1.000   0.900 |    1.000    0.920
 100 |   1.000   0.900 |    1.000    0.920
 200 |   1.000   0.859 |    1.000    0.900
 400 |   1.000   0.879 |    1.000    0.900


In [10]:
# === Bootstrap 95% CIs: ByT5 vs mT5, length-matched (paired over the same 50 BN items) ===
import numpy as np, os
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import f1_score

BASE = os.path.expanduser("~/Ayesha/Dyslexia")
# ByT5 reps already in memory (reps_en, reps_bn). Load mT5 reps (same df order -> pos_* index both).
mt5_en = np.load(f"{BASE}/dyslexia-mt5/representations/mt5_en_reps.npy")
mt5_bn = np.load(f"{BASE}/dyslexia-mt5/representations/mt5_bn_reps.npy")
REPS = {'byt5': (reps_en, reps_bn), 'mt5': (mt5_en, mt5_bn)}
print("reps:", {m:(e.shape, b.shape) for m,(e,b) in REPS.items()})

def fit_probe(Ren, L):
    pipe = make_pipeline(StandardScaler(),
                         LogisticRegression(max_iter=2000, C=1.0, solver='liblinear'))
    pipe.fit(Ren[pos_en, L, :], y_en_match)
    return pipe

def peak_layer(Ren, Rbn):
    best, bestL = -1, 0
    for L in range(Ren.shape[1]):
        p = fit_probe(Ren, L)
        f = f1_score(y_bn_match, p.predict(Rbn[pos_bn, L, :]), average='macro')
        if f > best: best, bestL = f, L
    return bestL

def ci(arr): return np.percentile(arr, [2.5, 50, 97.5])

# ---------- 1. TRANSFER F1 at each model's peak layer (paired bootstrap) ----------
peaks, base_pred, base_f1 = {}, {}, {}
for m,(Ren,Rbn) in REPS.items():
    L = peak_layer(Ren, Rbn); peaks[m] = L
    p = fit_probe(Ren, L)
    base_pred[m] = p.predict(Rbn[pos_bn, L, :])
    base_f1[m]   = f1_score(y_bn_match, base_pred[m], average='macro')

B = 5000; n = len(y_bn_match)
rng = np.random.default_rng(0)
boot = {m: [] for m in REPS}; diff = []
for _ in range(B):
    idx = rng.integers(0, n, n)
    yt = y_bn_match[idx]
    if len(np.unique(yt)) < 2: continue
    fs = {m: f1_score(yt, base_pred[m][idx], average='macro') for m in REPS}
    for m in REPS: boot[m].append(fs[m])
    diff.append(fs['byt5'] - fs['mt5'])

print("\n=== Cross-lingual transfer F1 (peak layer, 95% CI) ===")
for m in REPS:
    lo, md, hi = ci(boot[m])
    print(f"  {m:5s}  peak L{peaks[m]:>2d}  F1={base_f1[m]:.3f}  CI=[{lo:.3f}, {hi:.3f}]")
lo, md, hi = ci(diff)
sig = "SIGNIFICANT (excludes 0)" if lo > 0 else "not significant"
print(f"  ByT5 - mT5 = {base_f1['byt5']-base_f1['mt5']:+.3f}  CI=[{lo:+.3f}, {hi:+.3f}]  -> {sig}")

# ---------- 2. LOCALIZATION specificity (direct top-K zeroing, paired bootstrap) ----------
K = 20; N_RAND = 10
loc = {}   # m -> (pred_base, pred_top, [rand preds])
for m,(Ren,Rbn) in REPS.items():
    L = peaks[m]
    p = fit_probe(Ren, L)
    Xte = Rbn[pos_bn, L, :]
    ranked = np.argsort(np.abs(p.named_steps['logisticregression'].coef_[0]))[::-1]
    Xtop = Xte.copy(); Xtop[:, ranked[:K]] = 0.0
    rr = np.random.default_rng(123)
    rand_preds = []
    for _ in range(N_RAND):
        Xr = Xte.copy(); Xr[:, rr.choice(Xte.shape[1], K, replace=False)] = 0.0
        rand_preds.append(p.predict(Xr))
    loc[m] = (p.predict(Xte), p.predict(Xtop), rand_preds)

rng = np.random.default_rng(1)
spec_boot = {m: [] for m in REPS}; spec_diff = []
for _ in range(B):
    idx = rng.integers(0, n, n); yt = y_bn_match[idx]
    if len(np.unique(yt)) < 2: continue
    s = {}
    for m in REPS:
        pb, pt, rps = loc[m]
        base = f1_score(yt, pb[idx], average='macro')
        dtop = base - f1_score(yt, pt[idx], average='macro')
        drand = np.mean([base - f1_score(yt, rp[idx], average='macro') for rp in rps])
        s[m] = dtop - drand; spec_boot[m].append(s[m])
    spec_diff.append(s['mt5'] - s['byt5'])   # expect mT5 MORE localized

print(f"\n=== Localization specificity @top{K} (direct zeroing, 95% CI) ===")
print("  (positive = top units matter more than random = localized)")
for m in REPS:
    lo, md, hi = ci(spec_boot[m])
    print(f"  {m:5s}  spec={md:+.3f}  CI=[{lo:+.3f}, {hi:+.3f}]")
lo, md, hi = ci(spec_diff)
sig = "SIGNIFICANT (excludes 0)" if lo > 0 else "not significant"
print(f"  mT5 - ByT5 spec = {md:+.3f}  CI=[{lo:+.3f}, {hi:+.3f}]  -> {sig}")

reps: {'byt5': ((220, 37, 2560), (220, 37, 2560)), 'mt5': ((220, 25, 2048), (220, 25, 2048))}

=== Cross-lingual transfer F1 (peak layer, 95% CI) ===
  byt5   peak L29  F1=0.920  CI=[0.836, 0.980]
  mt5    peak L20  F1=0.599  CI=[0.449, 0.740]
  ByT5 - mT5 = +0.321  CI=[+0.153, +0.490]  -> SIGNIFICANT (excludes 0)

=== Localization specificity @top20 (direct zeroing, 95% CI) ===
  (positive = top units matter more than random = localized)
  byt5   spec=+0.012  CI=[+0.000, +0.038]
  mt5    spec=+0.187  CI=[+0.057, +0.319]
  mT5 - ByT5 spec = +0.175  CI=[+0.043, +0.308]  -> SIGNIFICANT (excludes 0)


In [11]:
# === Bootstrap 95% CIs: ALL 5 models, length-matched, paired over the same 50 BN items ===
import numpy as np, pandas as pd, os
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import f1_score

BASE = os.path.expanduser("~/Ayesha/Dyslexia")
REP_DIRS = {'xlmr': f"{BASE}/Dyslexia-XLM-R/representations",
            'llama': f"{BASE}/dyslexia-LLM/representations",
            'bloom': f"{BASE}/dyslexia-bloom/representations",
            'mt5':   f"{BASE}/dyslexia-mt5/representations"}
en_id_order = df_en['id'].values   # canonical df order (df_en/df_bn from Cells 4/5)
bn_id_order = df_bn['id'].values

def _ids(path):
    d = pd.read_csv(path); return (d['id'] if 'id' in d.columns else d.iloc[:, 0]).values

def load_aligned(m):
    if m == 'mt5':
        en = np.load(f"{REP_DIRS['mt5']}/mt5_en_reps.npy"); bn = np.load(f"{REP_DIRS['mt5']}/mt5_bn_reps.npy")
        en_ids = np.load(f"{REP_DIRS['mt5']}/mt5_ids_en.npy", allow_pickle=True)
        bn_ids = np.load(f"{REP_DIRS['mt5']}/mt5_ids_bn.npy", allow_pickle=True)
    else:
        en = np.load(f"{REP_DIRS[m]}/{m}_english_reps.npy"); bn = np.load(f"{REP_DIRS[m]}/{m}_bangla_reps.npy")
        en_ids = _ids(f"{REP_DIRS[m]}/{m}_english_meta.csv"); bn_ids = _ids(f"{REP_DIRS[m]}/{m}_bangla_meta.csv")
    ep = {v: i for i, v in enumerate(en_ids)}; bp = {v: i for i, v in enumerate(bn_ids)}
    return en[[ep[i] for i in en_id_order]], bn[[bp[i] for i in bn_id_order]]

REPS = {'byt5': (reps_en, reps_bn)}      # ByT5 already in df order, in memory
for m in ['xlmr', 'llama', 'bloom', 'mt5']:
    REPS[m] = load_aligned(m)
print("Loaded:", {m: (e.shape, b.shape) for m, (e, b) in REPS.items()})

def fit_probe(Ren, L):
    p = make_pipeline(StandardScaler(),
                      LogisticRegression(max_iter=2000, C=1.0, solver='liblinear'))
    p.fit(Ren[pos_en, L, :], y_en_match); return p

def peak_layer(Ren, Rbn):
    best, bestL = -1, 0
    for L in range(Ren.shape[1]):
        f = f1_score(y_bn_match, fit_probe(Ren, L).predict(Rbn[pos_bn, L, :]), average='macro')
        if f > best: best, bestL = f, L
    return bestL

ORDER = ['xlmr', 'llama', 'bloom', 'mt5', 'byt5']
K, N_RAND, B = 20, 10, 5000
peaks, base_pred, base_f1, loc = {}, {}, {}, {}
for m in ORDER:
    Ren, Rbn = REPS[m]; L = peak_layer(Ren, Rbn); peaks[m] = L
    p = fit_probe(Ren, L); Xte = Rbn[pos_bn, L, :]
    base_pred[m] = p.predict(Xte); base_f1[m] = f1_score(y_bn_match, base_pred[m], average='macro')
    ranked = np.argsort(np.abs(p.named_steps['logisticregression'].coef_[0]))[::-1]
    Xtop = Xte.copy(); Xtop[:, ranked[:K]] = 0.0
    rr = np.random.default_rng(123); rand_preds = []
    for _ in range(N_RAND):
        Xr = Xte.copy(); Xr[:, rr.choice(Xte.shape[1], K, replace=False)] = 0.0
        rand_preds.append(p.predict(Xr))
    loc[m] = (base_pred[m], p.predict(Xtop), rand_preds)

ci = lambda a: np.percentile(a, [2.5, 50, 97.5])
n = len(y_bn_match); rng = np.random.default_rng(0)
tf = {m: [] for m in ORDER}; sp = {m: [] for m in ORDER}
d_tf_bm, d_sp_mb = [], []
for _ in range(B):
    idx = rng.integers(0, n, n); yt = y_bn_match[idx]
    if len(np.unique(yt)) < 2: continue
    for m in ORDER:
        tf[m].append(f1_score(yt, base_pred[m][idx], average='macro'))
        pb, pt, rps = loc[m]
        base = f1_score(yt, pb[idx], average='macro')
        dtop = base - f1_score(yt, pt[idx], average='macro')
        drand = np.mean([base - f1_score(yt, rp[idx], average='macro') for rp in rps])
        sp[m].append(dtop - drand)
    d_tf_bm.append(tf['byt5'][-1] - tf['mt5'][-1])
    d_sp_mb.append(sp['mt5'][-1] - sp['byt5'][-1])

SURF = surf_bn
print(f"\n=== Transfer F1 (peak layer) & localization spec@{K}, 95% CI  [surface BN={SURF:.3f}] ===")
print(f"{'model':6s} {'pkL':>3s} | {'transferF1':>10s} {'95% CI':>16s} {'>surf?':>7s} | {'spec':>6s} {'95% CI':>16s} {'localized?':>10s}")
rows=[]
for m in ORDER:
    tlo,tmd,thi = ci(tf[m]); slo,smd,shi = ci(sp[m])
    surv = 'YES' if tlo > SURF else 'no'
    locd = 'YES' if slo > 0 else 'no'
    print(f"{m:6s} {peaks[m]:>3d} | {base_f1[m]:>10.3f} [{tlo:.3f},{thi:.3f}] {surv:>7s} | {smd:>+6.3f} [{slo:+.3f},{shi:+.3f}] {locd:>10s}")
    rows.append({'model':m,'peak_L':peaks[m],'transfer_f1':base_f1[m],'tf_lo':tlo,'tf_hi':thi,
                 'survives':surv=='YES','spec':smd,'spec_lo':slo,'spec_hi':shi,'localized':locd=='YES','surface_bn':SURF})
pd.DataFrame(rows).to_csv(f"{BASE}/Dyslexia-ByT5/results_bootstrap_ci_all_models.csv", index=False)

lo,md,hi = ci(d_tf_bm); print(f"\nByT5 - mT5 transfer = {base_f1['byt5']-base_f1['mt5']:+.3f}  CI=[{lo:+.3f},{hi:+.3f}]  -> {'SIGNIFICANT' if lo>0 else 'ns'}")
lo,md,hi = ci(d_sp_mb); print(f"mT5 - ByT5 spec     = {md:+.3f}  CI=[{lo:+.3f},{hi:+.3f}]  -> {'SIGNIFICANT' if lo>0 else 'ns'}")
print("\nSaved results_bootstrap_ci_all_models.csv")

Loaded: {'byt5': ((220, 37, 2560), (220, 37, 2560)), 'xlmr': ((220, 25, 1024), (220, 25, 1024)), 'llama': ((220, 33, 4096), (220, 33, 4096)), 'bloom': ((220, 31, 4096), (220, 31, 4096)), 'mt5': ((220, 25, 2048), (220, 25, 2048))}

=== Transfer F1 (peak layer) & localization spec@20, 95% CI  [surface BN=0.623] ===
model  pkL | transferF1           95% CI  >surf? |   spec           95% CI localized?
xlmr    16 |      0.859 [0.754,0.940]     YES | +0.135 [+0.046,+0.243]        YES
llama   31 |      0.635 [0.483,0.769]      no | -0.056 [-0.123,-0.008]         no
bloom   25 |      0.920 [0.839,0.980]     YES | +0.000 [+0.000,+0.000]         no
mt5     20 |      0.599 [0.449,0.740]      no | +0.186 [+0.058,+0.319]        YES
byt5    29 |      0.920 [0.836,0.980]     YES | +0.012 [+0.000,+0.038]         no

ByT5 - mT5 transfer = +0.321  CI=[+0.153,+0.490]  -> SIGNIFICANT
mT5 - ByT5 spec     = +0.174  CI=[+0.045,+0.307]  -> SIGNIFICANT

Saved results_bootstrap_ci_all_models.csv


In [12]:
# === Downstream cross-lingual screening-property classifier: train EN -> test BN ===
# Honest framing: predicts a dyslexia-relevant property of screening STIMULI cross-lingually
# (low-resource screening-item construction), NOT patient-level dyslexia screening.
import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import f1_score, roc_auc_score

LABEL = 'phonological_complexity'   # primary. Alt: set BINARIZE below for a binary AUC task.
BINARIZE = None                     # e.g. ('difficulty','hard') -> hard vs rest (gives ROC-AUC)

def get_y(df_lang):
    if BINARIZE:
        col, pos = BINARIZE; return (df_lang[col].astype(str) == pos).astype(int).values
    return df_lang[LABEL].astype(str).values

y_en_raw, y_bn_raw = get_y(df_en), get_y(df_bn)
le = LabelEncoder().fit(np.concatenate([y_en_raw, y_bn_raw]))
y_en, y_bn = le.transform(y_en_raw), le.transform(y_bn_raw)
classes = list(le.classes_)
print(f"LABEL={LABEL if not BINARIZE else BINARIZE}  classes={classes}")
print(f"EN dist={np.bincount(y_en).tolist()}  BN dist={np.bincount(y_bn).tolist()}")

def best_layer_en(Ren, y, nL):
    skf = StratifiedKFold(5, shuffle=True, random_state=42); best, bestL = -1, 0
    for L in range(nL):
        clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, C=1.0))
        s = cross_val_score(clf, Ren[:, L, :], y, cv=skf, scoring='f1_macro').mean()
        if s > best: best, bestL = s, L
    return bestL

def surface_feats(df_lang):
    out = []
    for _, r in df_lang.iterrows():
        w = str(r['word'])
        try: sc = float(r.get('syllable_count', 0))
        except: sc = 0.0
        out.append([len(w), len(set(w)), len(w.encode('utf-8')), sc])
    return np.array(out, float)

FEATS = {'surface': (surface_feats(df_en), surface_feats(df_bn))}
for m in ['xlmr','llama','bloom','mt5','byt5']:
    Ren, Rbn = REPS[m]; L = best_layer_en(Ren, y_en, Ren.shape[1])
    FEATS[m] = (Ren[:, L, :], Rbn[:, L, :]); print(f"  {m:6s} best EN layer L{L}")

def fit_eval(Xen, Xbn):
    clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, C=1.0)); clf.fit(Xen, y_en)
    pred, proba = clf.predict(Xbn), clf.predict_proba(Xbn)
    auc = roc_auc_score(y_bn, proba[:, 1]) if len(classes) == 2 else np.nan
    return pred, f1_score(y_bn, pred, average='macro'), auc

res = {k: fit_eval(*v) for k, v in FEATS.items()}
B, n = 5000, len(y_bn); rng = np.random.default_rng(0); ci = lambda a: np.percentile(a, [2.5, 50, 97.5])
boot = {k: [] for k in FEATS}; diff = {m: [] for m in ['xlmr','llama','bloom','mt5','byt5']}
for _ in range(B):
    idx = rng.integers(0, n, n)
    if len(np.unique(y_bn[idx])) < 2: continue
    s = {k: f1_score(y_bn[idx], res[k][0][idx], average='macro') for k in FEATS}
    for k in FEATS: boot[k].append(s[k])
    for m in diff: diff[m].append(s[m] - s['surface'])

print(f"\n=== Cross-lingual {LABEL if not BINARIZE else BINARIZE}: train EN -> test BN (macro-F1, 95% CI) ===")
for k in ['surface','xlmr','llama','bloom','mt5','byt5']:
    lo, md, hi = ci(boot[k]); auc = res[k][2]
    print(f"  {k:8s}  F1={res[k][1]:.3f}  CI=[{lo:.3f},{hi:.3f}]" + (f"  AUC={auc:.3f}" if not np.isnan(auc) else ""))
print("\n  LLM - surface (paired):")
for m in ['xlmr','llama','bloom','mt5','byt5']:
    lo, md, hi = ci(diff[m])
    print(f"    {m:6s}  Δ={res[m][1]-res['surface'][1]:+.3f}  CI=[{lo:+.3f},{hi:+.3f}]  {'BEATS surface' if lo>0 else 'ns'}")

LABEL=phonological_complexity  classes=['high', 'low', 'medium']
EN dist=[37, 96, 87]  BN dist=[37, 95, 88]
  xlmr   best EN layer L0
  llama  best EN layer L15
  bloom  best EN layer L0
  mt5    best EN layer L15
  byt5   best EN layer L10

=== Cross-lingual phonological_complexity: train EN -> test BN (macro-F1, 95% CI) ===
  surface   F1=0.374  CI=[0.301,0.441]
  xlmr      F1=0.306  CI=[0.247,0.367]
  llama     F1=0.275  CI=[0.220,0.327]
  bloom     F1=0.332  CI=[0.271,0.392]
  mt5       F1=0.477  CI=[0.403,0.544]
  byt5      F1=0.230  CI=[0.184,0.276]

  LLM - surface (paired):
    xlmr    Δ=-0.069  CI=[-0.155,+0.018]  ns
    llama   Δ=-0.100  CI=[-0.192,-0.008]  ns
    bloom   Δ=-0.042  CI=[-0.129,+0.045]  ns
    mt5     Δ=+0.102  CI=[+0.003,+0.200]  BEATS surface
    byt5    Δ=-0.144  CI=[-0.228,-0.057]  ns


In [13]:
# === Downstream binary screening: flag HARD items, train EN -> test BN (ROC-AUC) ===
import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import f1_score, roc_auc_score

y_en = (df_en['difficulty'].astype(str) == 'hard').astype(int).values
y_bn = (df_bn['difficulty'].astype(str) == 'hard').astype(int).values
print(f"HARD items -- EN={y_en.sum()}/{len(y_en)}  BN={y_bn.sum()}/{len(y_bn)}")

def best_layer_en(Ren, y, nL):
    skf = StratifiedKFold(5, shuffle=True, random_state=42); best, bestL = -1, 0
    for L in range(nL):
        clf = make_pipeline(StandardScaler(),
                            LogisticRegression(max_iter=2000, C=1.0, class_weight='balanced'))
        s = cross_val_score(clf, Ren[:, L, :], y, cv=skf, scoring='roc_auc').mean()
        if s > best: best, bestL = s, L
    return bestL

def surface_feats(df_lang):
    out = []
    for _, r in df_lang.iterrows():
        w = str(r['word'])
        try: sc = float(r.get('syllable_count', 0))
        except: sc = 0.0
        out.append([len(w), len(set(w)), len(w.encode('utf-8')), sc])
    return np.array(out, float)

FEATS = {'surface': (surface_feats(df_en), surface_feats(df_bn))}
for m in ['xlmr','llama','bloom','mt5','byt5']:
    Ren, Rbn = REPS[m]; L = best_layer_en(Ren, y_en, Ren.shape[1])
    FEATS[m] = (Ren[:, L, :], Rbn[:, L, :]); print(f"  {m:6s} best EN layer L{L}")

def fit_eval(Xen, Xbn):
    clf = make_pipeline(StandardScaler(),
                        LogisticRegression(max_iter=2000, C=1.0, class_weight='balanced'))
    clf.fit(Xen, y_en); proba = clf.predict_proba(Xbn)[:, 1]; pred = clf.predict(Xbn)
    return proba, pred, roc_auc_score(y_bn, proba), f1_score(y_bn, pred, average='macro')

res = {k: fit_eval(*v) for k, v in FEATS.items()}
B, n = 5000, len(y_bn); rng = np.random.default_rng(0); ci = lambda a: np.percentile(a, [2.5, 50, 97.5])
bootA = {k: [] for k in FEATS}; diffA = {m: [] for m in ['xlmr','llama','bloom','mt5','byt5']}
for _ in range(B):
    idx = rng.integers(0, n, n)
    if len(np.unique(y_bn[idx])) < 2: continue
    a = {k: roc_auc_score(y_bn[idx], res[k][0][idx]) for k in FEATS}
    for k in FEATS: bootA[k].append(a[k])
    for m in diffA: diffA[m].append(a[m] - a['surface'])

print(f"\n=== Flag HARD items: train EN -> test BN  (ROC-AUC, 95% CI) ===")
for k in ['surface','xlmr','llama','bloom','mt5','byt5']:
    lo, md, hi = ci(bootA[k])
    print(f"  {k:8s}  AUC={res[k][2]:.3f}  CI=[{lo:.3f},{hi:.3f}]  (macroF1={res[k][3]:.3f})")
print("\n  LLM - surface AUC (paired):")
for m in ['xlmr','llama','bloom','mt5','byt5']:
    lo, md, hi = ci(diffA[m])
    print(f"    {m:6s}  Δ={res[m][2]-res['surface'][2]:+.3f}  CI=[{lo:+.3f},{hi:+.3f}]  {'BEATS surface' if lo>0 else 'ns'}")

HARD items -- EN=46/220  BN=32/220
  xlmr   best EN layer L22
  llama  best EN layer L31
  bloom  best EN layer L20
  mt5    best EN layer L13
  byt5   best EN layer L14

=== Flag HARD items: train EN -> test BN  (ROC-AUC, 95% CI) ===
  surface   AUC=0.795  CI=[0.675,0.898]  (macroF1=0.755)
  xlmr      AUC=0.762  CI=[0.677,0.841]  (macroF1=0.643)
  llama     AUC=0.542  CI=[0.436,0.646]  (macroF1=0.480)
  bloom     AUC=0.774  CI=[0.672,0.863]  (macroF1=0.671)
  mt5       AUC=0.878  CI=[0.804,0.938]  (macroF1=0.751)
  byt5      AUC=0.797  CI=[0.720,0.865]  (macroF1=0.397)

  LLM - surface AUC (paired):
    xlmr    Δ=-0.033  CI=[-0.149,+0.089]  ns
    llama   Δ=-0.253  CI=[-0.388,-0.117]  ns
    bloom   Δ=-0.021  CI=[-0.112,+0.070]  ns
    mt5     Δ=+0.083  CI=[-0.005,+0.187]  ns
    byt5    Δ=+0.001  CI=[-0.106,+0.118]  ns


In [14]:
# === Downstream: cross-lingual dyslexia ERROR-TYPE classification, train EN -> test BN ===
# Most dyslexia-specific label. Restricted to error types present in BOTH languages.
import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import f1_score

et_en = df_en['error_type_targeted'].astype(str)
et_bn = df_bn['error_type_targeted'].astype(str)
shared = sorted(set(et_en.unique()) & set(et_bn.unique()))
print(f"Shared error types ({len(shared)}): {shared}")
print(f"Dropped EN-only: {sorted(set(et_en)-set(shared))}  BN-only: {sorted(set(et_bn)-set(shared))}")

en_mask = et_en.isin(shared).values; bn_mask = et_bn.isin(shared).values
le = LabelEncoder().fit(shared)
y_en = le.transform(et_en[en_mask]); y_bn = le.transform(et_bn[bn_mask])
print(f"EN n={en_mask.sum()}  BN n={bn_mask.sum()}  classes={len(shared)}")

def best_layer_en(Ren, y, nL):
    skf = StratifiedKFold(5, shuffle=True, random_state=42); best, bestL = -1, 0
    for L in range(nL):
        clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, C=1.0))
        s = cross_val_score(clf, Ren[:, L, :], y, cv=skf, scoring='f1_macro').mean()
        if s > best: best, bestL = s, L
    return bestL

def surface_feats(df_lang, mask):
    out = []
    for _, r in df_lang[mask].iterrows():
        w = str(r['word'])
        try: sc = float(r.get('syllable_count', 0))
        except: sc = 0.0
        out.append([len(w), len(set(w)), len(w.encode('utf-8')), sc])
    return np.array(out, float)

FEATS = {'surface': (surface_feats(df_en, en_mask), surface_feats(df_bn, bn_mask))}
for m in ['xlmr','llama','bloom','mt5','byt5']:
    Ren, Rbn = REPS[m]; Ren_f, Rbn_f = Ren[en_mask], Rbn[bn_mask]
    L = best_layer_en(Ren_f, y_en, Ren_f.shape[1])
    FEATS[m] = (Ren_f[:, L, :], Rbn_f[:, L, :]); print(f"  {m:6s} best EN layer L{L}")

def fit_eval(Xen, Xbn):
    clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, C=1.0))
    clf.fit(Xen, y_en); return clf.predict(Xbn)

preds = {k: fit_eval(*v) for k, v in FEATS.items()}
f1s = {k: f1_score(y_bn, preds[k], average='macro') for k in FEATS}
B, n = 5000, len(y_bn); rng = np.random.default_rng(0); ci = lambda a: np.percentile(a, [2.5, 50, 97.5])
boot = {k: [] for k in FEATS}; diff = {m: [] for m in ['xlmr','llama','bloom','mt5','byt5']}
for _ in range(B):
    idx = rng.integers(0, n, n)
    if len(np.unique(y_bn[idx])) < 2: continue
    s = {k: f1_score(y_bn[idx], preds[k][idx], average='macro') for k in FEATS}
    for k in FEATS: boot[k].append(s[k])
    for m in diff: diff[m].append(s[m] - s['surface'])

print(f"\n=== Cross-lingual error-type ({len(shared)}-class): train EN -> test BN (macro-F1, 95% CI) ===")
for k in ['surface','xlmr','llama','bloom','mt5','byt5']:
    lo, md, hi = ci(boot[k]); print(f"  {k:8s}  F1={f1s[k]:.3f}  CI=[{lo:.3f},{hi:.3f}]")
print("\n  LLM - surface (paired):")
for m in ['xlmr','llama','bloom','mt5','byt5']:
    lo, md, hi = ci(diff[m])
    verdict = 'BEATS surface' if lo > 0 else ('WORSE (sig)' if hi < 0 else 'ns')
    print(f"    {m:6s}  Δ={f1s[m]-f1s['surface']:+.3f}  CI=[{lo:+.3f},{hi:+.3f}]  {verdict}")

Shared error types (8): ['multisyllabic_breakdown', 'none', 'phoneme_grapheme_mapping', 'pseudoword_decoding', 'rhyme_judgment', 'sentence_processing', 'voiced_unvoiced_confusion', 'vowel_substitution']
Dropped EN-only: ['substitution']  BN-only: ['aspiration_confusion']
EN n=206  BN n=204  classes=8
  xlmr   best EN layer L11
  llama  best EN layer L17
  bloom  best EN layer L12
  mt5    best EN layer L17
  byt5   best EN layer L31

=== Cross-lingual error-type (8-class): train EN -> test BN (macro-F1, 95% CI) ===
  surface   F1=0.258  CI=[0.209,0.302]
  xlmr      F1=0.310  CI=[0.243,0.370]
  llama     F1=0.027  CI=[0.012,0.047]
  bloom     F1=0.257  CI=[0.211,0.299]
  mt5       F1=0.294  CI=[0.235,0.348]
  byt5      F1=0.322  CI=[0.281,0.358]

  LLM - surface (paired):
    xlmr    Δ=+0.052  CI=[-0.028,+0.129]  ns
    llama   Δ=-0.231  CI=[-0.278,-0.180]  WORSE (sig)
    bloom   Δ=-0.001  CI=[-0.061,+0.059]  ns
    mt5     Δ=+0.036  CI=[-0.041,+0.108]  ns
    byt5    Δ=+0.064  CI=[+0.

In [15]:
# === Figure: bootstrap-CI forest plot (transfer + localization), all 5 models ===
import pandas as pd, numpy as np, os
import matplotlib.pyplot as plt

BASE = os.path.expanduser("~/Ayesha/Dyslexia")
df = pd.read_csv(f"{BASE}/Dyslexia-ByT5/results_bootstrap_ci_all_models.csv")
SURF = float(df['surface_bn'].iloc[0])

COLORS = {'xlmr':'#2F6FA6','bloom':'#3A9A55','mt5':'#8A5CB0','llama':'#C33F3C','byt5':'#E07B00'}
LABELS = {'xlmr':'XLM-R','bloom':'BLOOM','mt5':'mT5','llama':'Llama','byt5':'ByT5'}

df = df.sort_values('transfer_f1', ascending=True).reset_index(drop=True)  # highest at top
y = np.arange(len(df)); cols = [COLORS[m] for m in df['model']]

fig, (axA, axB) = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)

# Panel A: transfer F1
errA = np.vstack([df['transfer_f1']-df['tf_lo'], df['tf_hi']-df['transfer_f1']])
for i in range(len(df)):
    axA.errorbar(df['transfer_f1'][i], y[i], xerr=errA[:, i:i+1], fmt='o', ms=8,
                 color=cols[i], ecolor=cols[i], elinewidth=2.5, capsize=4, capthick=2)
    axA.annotate(f"{df['transfer_f1'][i]:.2f}", (df['transfer_f1'][i], y[i]), textcoords="offset points",
                 xytext=(0, 10), ha='center', fontsize=9, color=cols[i], fontweight='bold')
axA.axvline(SURF, color='#888780', ls='--', lw=1.5)
axA.text(SURF+0.006, -0.45, f'surface {SURF:.2f}', color='#5F5E5A', fontsize=9, va='center')
axA.set_xlabel('EN→BN transfer F1 (macro, length-matched)', fontsize=10)
axA.set_title('Cross-lingual transfer\nright of line = survives surface control', fontsize=10.5, fontweight='bold')
axA.set_xlim(0.4, 1.0); axA.grid(axis='x', alpha=0.3)

# Panel B: localization specificity
errB = np.vstack([df['spec']-df['spec_lo'], df['spec_hi']-df['spec']])
for i in range(len(df)):
    axB.errorbar(df['spec'][i], y[i], xerr=errB[:, i:i+1], fmt='o', ms=8,
                 color=cols[i], ecolor=cols[i], elinewidth=2.5, capsize=4, capthick=2)
    axB.annotate(f"{df['spec'][i]:+.2f}", (df['spec'][i], y[i]), textcoords="offset points",
                 xytext=(0, 10), ha='center', fontsize=9, color=cols[i], fontweight='bold')
axB.axvline(0, color='#888780', ls='-', lw=1.5)
axB.set_xlabel('localization specificity (top-20 − random, BN)', fontsize=10)
axB.set_title('Unit localization\nexcludes 0 = localized; at 0 = distributed', fontsize=10.5, fontweight='bold')
axB.set_xlim(-0.15, 0.35); axB.grid(axis='x', alpha=0.3)

axA.set_yticks(y); axA.set_yticklabels([LABELS[m] for m in df['model']], fontsize=11)
axA.set_ylim(-0.6, len(df)-0.4)
fig.suptitle('Byte-level ByT5 vs subword models: cross-lingual transfer and its mechanism (95% bootstrap CIs, n=50)',
             fontsize=12, fontweight='bold', y=1.03)
fig.tight_layout()
out = f"{BASE}/Dyslexia-ByT5/plots_bootstrap_ci_forest"
fig.savefig(out+".png", dpi=200, bbox_inches='tight'); fig.savefig(out+".pdf", bbox_inches='tight')
plt.close(fig)
print("Saved", out+".png / .pdf")

Saved /home/accesx-denied/Ayesha/Dyslexia/Dyslexia-ByT5/plots_bootstrap_ci_forest.png / .pdf
